# 🎵 Love Songs: Prompt Chaining Notebook
**CompLit 126x — Love in Context**

This notebook walks you through the full prompt chain for the Love Songs assignment.

**The problem:** A spider chart (trait scores) helps you *compare* poems, but it's a lossy compression — numbers can't capture what makes a poem feel like itself. This notebook shows you how to build a *prompt chain* that recovers what the numbers miss.

**The chain:**
1. **Analyze** poems → extract trait scores (the spider chart)
2. **Extract** poetic texture → specific images, phrases, and formal moves
3. **Synthesize** a voice profile across your poem pool
4. **Generate** lyrics with full context
5. **Compare** to the scores-only version (what you saw in the web tool)

---
**Run the cells in order.** For Steps 1–3, repeat for each poem in your pool (aim for 3+).

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import re
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)

# Initialize your poem pool (stores analyses across poems)
poem_pool = []

print(f"✓ Client ready | poem_pool initialized (empty)")

---
## Step 0: Define Your Traits

Before analyzing poems, decide which traits you want to track across your poet's work. Edit the list below — 4–6 traits works well. You'll use the same traits for every poem in your pool.

In [ ]:
# ── Your traits ──────────────────────────────────────────────────────────────
# Customize these to capture what matters most for your chosen poet.

traits = [
    {"name": "Melancholy",     "description": "sadness, longing, or wistfulness"},
    {"name": "Romanticism",    "description": "love, passion, or deep emotional connection"},
    {"name": "Nature Imagery", "description": "the natural world as metaphor, symbol, or setting"},
    {"name": "Mortality",      "description": "death, time passing, or impermanence"},
    {"name": "Optimism",       "description": "hopeful outlook, positive resolution, or uplift"},
]

print(f"✓ {len(traits)} traits defined:")
for t in traits:
    print(f"  · {t['name']}: {t['description']}")

---
## Step 1: Add a Poem

Paste a poem in the cell below. Then run **this cell → Analyze → Extract → Save**.
Repeat for each poem in your pool. Aim for at least 3.

In [ ]:
# ── Paste your poem here ─────────────────────────────────────────────────────
# Replace the placeholder with the full text of one poem.

POEM = """
Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance, or nature's changing course untrimm'd;
But thy eternal summer shall not fade,
Nor lose possession of that fair thou ow'st;
Nor shall death brag thou wander'st in his shade,
When in eternal lines to time thou grow'st:
So long as men can breathe, or eyes can see,
So long lives this, and this gives life to thee.
"""

POEM_TITLE = "Sonnet 18"           # ← change this
POET_NAME  = "William Shakespeare" # ← change this (keep consistent across poems)

print(f"Poem loaded: '{POEM_TITLE}' by {POET_NAME} ({len(POEM.split())} words)")

### Analyze: Extract Trait Scores

In [ ]:
# ── Get trait scores ──────────────────────────────────────────────────────────
# This is the spider chart step — the same thing the web tool does.

trait_list = "\n".join(f"- {t['name']}: {t['description']}" for t in traits)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": f"""Score this poem on each trait from 1 to 10.
Return your answer as JSON in this exact format:
{{"analysis": [{{"trait": "trait name", "score": 7, "reasoning": "one sentence"}}]}}

Poem:
\"\"\"
{POEM}
\"\"\"

Traits:
{trait_list}"""
    }]
)

# Parse JSON from the response
raw = response.choices[0].message.content
try:
    scores_data = json.loads(raw)
except json.JSONDecodeError:
    match = re.search(r'\{[\s\S]*\}', raw)
    scores_data = json.loads(match.group()) if match else {"analysis": []}

# Display as a text bar chart
print(f"Scores for: {POEM_TITLE}\n{'─'*48}")
for item in scores_data["analysis"]:
    filled = "█" * int(item["score"])
    empty  = "░" * (10 - int(item["score"]))
    print(f"  {item['trait']:<20} {filled}{empty}  {item['score']}/10")
    print(f"  → {item['reasoning']}\n")

### Extract: Poetic Texture

This is the step the spider chart skips. Instead of scores, we're asking the model to identify the *specific language* — the images, rhythms, and formal moves — that make this poem feel like itself.

In [ ]:
# ── Extract poetic texture ────────────────────────────────────────────────────

texture_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": f"""Read this poem carefully and extract its poetic texture.

1. **Characteristic phrases**: Quote 3–5 specific lines or phrases most distinctive of this voice. Use the actual text.
2. **Formal moves**: How does the poem open? What turn does it take? How does it close?
3. **Emotional arc**: What feeling does it begin in, and where does it arrive?
4. **Sound and rhythm**: Any recurring sounds, rhythmic patterns, or structural habits?

Be specific. Quote the actual text. Don't summarize — show.

Poem:
\"\"\"
{POEM}
\"\"\""""
    }]
)

texture = texture_response.choices[0].message.content
print(f"Texture of: {POEM_TITLE}\n{'─'*48}")
print(texture)

### Save to Pool

In [ ]:
# ── Save this poem's analysis to the pool ─────────────────────────────────────
# Run this cell after Analyze + Extract above.
# Then go back to Step 1 and add another poem.

poem_pool.append({
    "title":   POEM_TITLE,
    "poet":    POET_NAME,
    "poem":    POEM,
    "scores":  scores_data["analysis"],
    "texture": texture,
})

print(f"✓ Saved '{POEM_TITLE}'")
print(f"  Pool: {[p['title'] for p in poem_pool]}")
if len(poem_pool) < 3:
    print(f"\n  → Go back to Step 1 and add another poem ({3 - len(poem_pool)} more recommended).")
else:
    print(f"\n  → Good pool size. Continue to Step 2 below.")

---
## Step 2: Synthesize a Voice Profile

Once you have 2+ poems in your pool, run this cell. It averages the scores across your poems and asks the model to synthesize a single voice profile from all the texture analyses.

In [ ]:
# ── Synthesize voice profile ──────────────────────────────────────────────────

if len(poem_pool) < 2:
    print("⚠  Add at least 2 poems first (Steps 1–Save above).")
else:
    # Average scores across all poems in pool
    trait_totals = {}
    for entry in poem_pool:
        for item in entry["scores"]:
            trait_totals.setdefault(item["trait"], []).append(item["score"])

    averaged = {t: sum(s) / len(s) for t, s in trait_totals.items()}

    print(f"Averaged scores across {len(poem_pool)} poems\n{'─'*48}")
    for trait, score in averaged.items():
        filled = "█" * int(score)
        empty  = "░" * (10 - int(score))
        print(f"  {trait:<20} {filled}{empty}  {score:.1f}/10")

    # Synthesize voice profile from all texture analyses
    all_textures = "\n\n---\n\n".join(
        f"From '{e['title']}':\n{e['texture']}" for e in poem_pool
    )

    synth_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": f"""I've analyzed {len(poem_pool)} poems by {poem_pool[0]['poet']}.

Here are the texture analyses:

{all_textures}

Write a 150-word voice profile for this poet: their characteristic imagery,
recurring themes, emotional territory, and formal habits.
Be specific — quote actual phrases from the poems where possible."""
        }]
    )

    voice_profile = synth_response.choices[0].message.content

    print(f"\nVoice profile for {poem_pool[0]['poet']}\n{'─'*48}")
    print(voice_profile)

---
## Step 3: Generate Lyrics with Full Context

Now we generate using the full chain — scores + texture + voice profile. This is the prompt chaining approach.

After running this, run **Step 4** (the scores-only version) and compare the two outputs.

In [ ]:
# ── Generate lyrics with full context ────────────────────────────────────────

score_summary = "\n".join(f"- {t}: {s:.1f}/10" for t, s in averaged.items())

gen_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": f"""Write a new love song inspired by {poem_pool[0]['poet']}'s style.

--- Emotional qualities (averaged from poem pool, 1–10 scale) ---
{score_summary}

--- Voice profile ---
{voice_profile}

--- Specific texture and phrases from the poems ---
{all_textures}

Write original song lyrics: a verse, a chorus, and a bridge.
Capture this poet's sensibility. Use their characteristic imagery and emotional range.
Do NOT copy their lines — write something new that sounds like it comes from the same place."""
    }]
)

lyrics_chained = gen_response.choices[0].message.content

print("═" * 60)
print("LYRICS — full chain (scores + texture + voice profile)")
print("═" * 60)
print(lyrics_chained)

---
## Step 4: Compare — Scores Only vs. Full Chain

This cell generates lyrics using *only* the averaged scores — no poem text, no texture. This is what the web tool produces.

Compare the two versions:
- Which uses more concrete imagery?
- Which sounds more like your poet?
- What did the chain add that the scores alone couldn't?

These observations are the core of your essay.

In [ ]:
# ── Scores-only version (for comparison) ─────────────────────────────────────

scores_only_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": f"""Write a love song with the following trait scores (scale of 1–10):

{score_summary}

Write complete song lyrics with a verse, a chorus, and a bridge."""
    }]
)

print("═" * 60)
print("LYRICS — scores only (no poem text)")
print("═" * 60)
print(scores_only_response.choices[0].message.content)

---
## Going Further

The chain above is a starting point. Here are variations to try — each one makes a good experiment to write about in your essay.

**Experiment 1: Add a revision step.**
Take the chain output and send it back with specific feedback:
> *"The chorus is too abstract. Revise it with a concrete image from the poems."*

**Experiment 2: Change what Step 2 extracts.**
Try asking for meter and rhyme scheme, or the grammar of the sentences, or the ratio of abstract to concrete language. Each gives the model a different lens.

**Experiment 3: Give the model the actual poem text.**
In Step 3, pass in the full poems instead of (or alongside) the texture analysis. Does the output get better? Does it start copying too directly?

**Experiment 4: Generate multiple songs.**
For your album, run Step 3 multiple times with different instructions:
> *"Write a ballad"* / *"Write in the style of a contemporary pop song"* / *"Write in second person"*

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet